# BDC Satria Data 2026 — OOF Collection (5-Fold) untuk Stacking
## Model: ViT-B/16 SigLIP2 + Swin-Base V2 + ConvNeXt V2-Base

**Tujuan:** kumpulkan probabilitas out-of-fold (OOF) dari 3 model terbaik kamu, dipakai sebagai bahan latih meta-learner (Tahap 2 workflow stacking).

**Beda dari notebook eksplorasi sebelumnya:**
- `StratifiedKFold(n_splits=5)` — bukan `StratifiedShuffleSplit`. Partisi bersih, tiap gambar tepat 1x jadi validation.
- **Simpan probabilitas SEMUA gambar** (benar maupun salah) — bukan cuma yang salah.
- **5-fold** (bukan 3) — OOF lebih halus, tapi berarti training 3 model x 5 fold = 15 run total. Realistis makan waktu belasan jam. Kalau ternyata terlalu lama, notebook ini bisa dihentikan di tengah dan diteruskan nanti (checkpoint per fold disimpan).
- **Preprocessing val/test diperbaiki**: `Resize(sisi pendek) + CenterCrop` (bukan `Resize` langsung ke kotak) — menghindari distorsi "gepeng" yang kita bahas sebelumnya. Ini PERTAMA KALINYA fix ini diterapkan di kode kamu.

**Windows + num_workers**: pakai pola `%%writefile dataset_utils.py` (sama seperti 2 notebook terakhir) supaya `NUM_WORKERS=8` bisa dipakai tanpa error pickling.

**Output notebook ini:** `oof_predictions_wide.csv` — 1 baris per gambar, kolom probabilitas 3 model (9 kolom total), siap dipakai melatih meta-learner.

## 1. Setup & Konfigurasi

In [ ]:
from __future__ import annotations
import os
import gc
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from torchvision import models as tv_models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

try:
    import timm
except ImportError:
    raise ImportError("timm belum terinstall. Jalankan: pip install timm")

In [ ]:
# === PATH DATASET ===
DATA_ROOT = Path(r"C:\Users\MyPC PRO\Downloads\BDC2026")
TRAIN_DIR = DATA_ROOT / "train"
MODELS_DIR = DATA_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = DATA_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

CLASS_FOLDERS = ["0_Recyclable", "1_Electronic", "2_Organic"]
CLASS_NAMES = ["Recyclable", "Electronic", "Organic"]
NUM_CLASSES = len(CLASS_NAMES)

# === MODEL LIST -- 3 model terbaik kamu ===
MODEL_LIST = [
    {"key": "siglip2_base",  "type": "timm", "timm_name": "vit_base_patch16_siglip_224.v2_webli", "img_size": 224, "batch_size": 32},
    {"key": "swinv2_base",   "type": "timm", "timm_name": "swinv2_base_window12to24_192to384.ms_in22k_ft_in1k", "img_size": 384, "batch_size": 16},
    {"key": "convnextv2_base","type": "timm", "timm_name": "convnextv2_base.fcmae_ft_in22k_in1k", "img_size": 224, "batch_size": 32},
]

# === CONFIG TRAINING (RTX 3060 12.9GB) ===
NUM_EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 4
N_FOLDS = 5              # 5-fold sesuai keputusan -- OOF lebih halus, tapi 3 model x 5 fold = 15 run
SEED = 42
NUM_WORKERS = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"N_FOLDS = {N_FOLDS} | Model dalam antrian: {len(MODEL_LIST)}")
for m in MODEL_LIST:
    print(f"  - {m['key']} @ res {m['img_size']}, batch {m['batch_size']}")
print(f"Total run: {len(MODEL_LIST)} model x {N_FOLDS} fold = {len(MODEL_LIST) * N_FOLDS} run")

## 2. Tulis `dataset_utils.py` (WAJIB di-run sebelum cell training)

**PERBAIKAN PENTING di sini:** transform validasi sekarang pakai `Resize(sisi pendek) + CenterCrop` — bukan `Resize` langsung ke kotak. Ini menghindari distorsi "gepeng" yang dibahas sebelumnya. Transform training tetap `RandomResizedCrop` seperti biasa.

In [ ]:
%%writefile dataset_utils.py
"""Dataset + transforms untuk BDC2026 -- versi OOF Collection.

Perbaikan dari notebook sebelumnya: transform val/test pakai Resize(sisi pendek)
+ CenterCrop (bukan Resize langsung ke kotak) -- menghindari distorsi aspect ratio.
"""
from __future__ import annotations

from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms as T


class TrashDataset(Dataset):
    def __init__(self, df, transform=None, fallback_size: int = 224):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.fallback_size = fallback_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row["path"]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (self.fallback_size, self.fallback_size))
        if self.transform:
            img = self.transform(img)
        return img, int(row["label"]), row["path"]


def build_transforms(mean, std, img_size: int, train: bool):
    if train:
        resize_to = int(img_size * 1.14)
        return T.Compose([
            T.Resize((resize_to, resize_to)),
            T.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
            T.ToTensor(),
            T.Normalize(mean=mean, std=std),
        ])
    # FIX: val/test pakai Resize(sisi pendek) + CenterCrop -- jaga rasio aspek, tidak gepeng.
    resize_short_side = int(img_size * 1.14)
    return T.Compose([
        T.Resize(resize_short_side),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])

## 3. Import dari `dataset_utils.py`

In [ ]:
import importlib
import dataset_utils
importlib.reload(dataset_utils)

from dataset_utils import TrashDataset, build_transforms

print("dataset_utils.py berhasil di-import (versi dengan fix Resize+CenterCrop)")

## 4. Index Dataset dari Folder

In [ ]:
def index_dataset(train_dir: Path, class_folders: list[str]):
    records = []
    for label_idx, folder_name in enumerate(class_folders):
        folder = train_dir / folder_name
        if not folder.is_dir():
            raise FileNotFoundError(f"Folder tidak ditemukan: {folder}")
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        files = [p for p in folder.iterdir() if p.suffix.lower() in exts]
        for p in files:
            records.append({"path": str(p), "label": label_idx, "class_name": CLASS_NAMES[label_idx]})
    return pd.DataFrame(records)

full_df = index_dataset(TRAIN_DIR, CLASS_FOLDERS)
print(f"Total citra ter-index: {len(full_df)}")
print(full_df["class_name"].value_counts())

## 5. Model Builder (generik: timm)

In [ ]:
def build_model(model_cfg: dict, num_classes: int):
    timm_name = model_cfg["timm_name"]
    model = timm.create_model(timm_name, pretrained=True, num_classes=num_classes)

    for p in model.parameters():
        p.requires_grad = False

    head_module = None
    for attr in ["head", "classifier", "fc"]:
        if hasattr(model, attr):
            candidate = getattr(model, attr)
            if isinstance(candidate, nn.Module):
                head_module = candidate
                break
    if head_module is None:
        raise AttributeError(f"Head classifier tidak ketemu untuk {timm_name}")

    for p in head_module.parameters():
        p.requires_grad = True

    data_cfg = timm.data.resolve_model_data_config(model)
    mean, std = data_cfg["mean"], data_cfg["std"]

    return model.to(DEVICE), mean, std

## 6. Train & Evaluate (mengembalikan probabilitas SEMUA sampel)

In [ ]:
scaler = GradScaler("cuda", enabled=(DEVICE == "cuda"))


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_paths, all_probs = [], [], [], []
    for imgs, labels, paths in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        running_loss += loss.item()

        probs = F.softmax(outputs.float(), dim=1)
        preds = probs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_paths.extend(paths)
        all_probs.extend(probs.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1, np.array(all_labels), np.array(all_preds), all_paths, np.array(all_probs)

## 7. Loop Utama — 3 Model x 5-Fold StratifiedKFold, Kumpulkan OOF SEMUA Gambar

**Catatan resiliensi:** loop ini bisa berjalan lama (estimasi belasan jam). Checkpoint tiap fold disimpan ke disk (`MODELS_DIR`), dan hasil per fold langsung ditulis ke `fold_summary` — kalau notebook terputus di tengah, kamu bisa cek `oof_records` sejauh mana sudah terkumpul dan melanjutkan secara manual (edit `MODEL_LIST` untuk skip model/fold yang sudah selesai).

In [ ]:
oof_records = []
fold_summary = []

for model_cfg in MODEL_LIST:
    model_key = model_cfg["key"]
    img_size = model_cfg["img_size"]
    batch_size = model_cfg["batch_size"]
    print(f"\n{'#'*70}\n### MODEL: {model_key} (StratifiedKFold, n_splits={N_FOLDS})\n{'#'*70}")

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    for fold, (train_idx, val_idx) in enumerate(skf.split(full_df, full_df["label"]), start=1):
        print(f"\n{'='*64}\n---> [{model_key}] FOLD {fold}/{N_FOLDS} <---\n{'='*64}")
        train_df = full_df.iloc[train_idx]
        val_df = full_df.iloc[val_idx]
        print(f"  train={len(train_df)} val={len(val_df)}")

        weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["label"].values)
        class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

        model, mean, std = build_model(model_cfg, NUM_CLASSES)
        if fold == 1:
            n_total = sum(p.numel() for p in model.parameters())
            n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  Total params: {n_total:,} | Trainable (head): {n_train:,}")

        train_tf = build_transforms(mean, std, img_size, train=True)
        val_tf = build_transforms(mean, std, img_size, train=False)

        train_loader = DataLoader(TrashDataset(train_df, train_tf, fallback_size=img_size),
                                  batch_size=batch_size, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
        val_loader = DataLoader(TrashDataset(val_df, val_tf, fallback_size=img_size),
                                batch_size=batch_size, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                      lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

        best_f1 = 0.0
        epochs_no_improve = 0
        best_path = MODELS_DIR / f"oof_{model_key}_fold{fold}.pth"

        for epoch in range(1, NUM_EPOCHS + 1):
            tr_loss, tr_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
            val_loss, val_f1, _, _, _, _ = evaluate(model, val_loader, criterion)
            scheduler.step(val_f1)

            marker = ""
            if val_f1 > best_f1:
                best_f1 = val_f1
                epochs_no_improve = 0
                torch.save(model.state_dict(), best_path)
                marker = " [*] best"
            else:
                epochs_no_improve += 1

            print(f"  Epoch {epoch:2d}/{NUM_EPOCHS} | train_loss {tr_loss:.4f} F1 {tr_f1:.4f} | "
                  f"val_loss {val_loss:.4f} F1 {val_f1:.4f}{marker}")

            if epochs_no_improve >= PATIENCE:
                print(f"  Early stopping.")
                break

        model.load_state_dict(torch.load(best_path, weights_only=True))
        _, final_f1, y_true, y_pred, paths, probs = evaluate(model, val_loader, criterion)
        print(f"  >> [{model_key}] Fold {fold} best macro F1: {best_f1:.4f} | re-eval F1: {final_f1:.4f}")

        fold_summary.append({"model": model_key, "fold": fold, "best_macro_f1": best_f1})

        for i, path in enumerate(paths):
            p = probs[i]
            oof_records.append({
                "model": model_key,
                "fold": fold,
                "filepath": path,
                "true_label": CLASS_NAMES[y_true[i]],
                "predicted_label": CLASS_NAMES[y_pred[i]],
                "is_correct": bool(y_true[i] == y_pred[i]),
                "prob_recyclable": float(p[0]),
                "prob_electronic": float(p[1]),
                "prob_organic": float(p[2]),
            })

        # Simpan progress inkremental -- kalau terputus, ini masih ada di disk
        pd.DataFrame(oof_records).to_csv(OUTPUT_DIR / "oof_predictions_long_INPROGRESS.csv", index=False)

        del model, train_loader, val_loader, optimizer, scheduler
        gc.collect()
        torch.cuda.empty_cache()

## 8. Ringkasan Fold & Simpan OOF Predictions (Long Format)

In [ ]:
summary_df = pd.DataFrame(fold_summary)
print("Ringkasan OOF per model (5-fold):\n")
for model_key in summary_df["model"].unique():
    sub = summary_df[summary_df["model"] == model_key]
    f1s = sub["best_macro_f1"].values
    print(f"  {model_key}: rata-rata {np.mean(f1s):.4f} (+/- {np.std(f1s):.4f})")
    for _, row in sub.iterrows():
        print(f"    Fold {int(row['fold'])}: {row['best_macro_f1']:.4f}")

oof_df = pd.DataFrame(oof_records)
oof_df.to_csv(OUTPUT_DIR / "oof_predictions_long.csv", index=False)
print(f"\n[SAVED] oof_predictions_long.csv ({len(oof_df)} baris)")

## 9. Pivot ke Format Wide (1 baris per gambar, 9 kolom probabilitas) — Siap untuk Meta-Learner

In [ ]:
pivot_frames = []
for model_key in oof_df["model"].unique():
    sub = oof_df[oof_df["model"] == model_key][["filepath", "true_label", "prob_recyclable", "prob_electronic", "prob_organic"]].copy()
    sub = sub.rename(columns={
        "prob_recyclable": f"{model_key}_prob_recyclable",
        "prob_electronic": f"{model_key}_prob_electronic",
        "prob_organic": f"{model_key}_prob_organic",
    })
    pivot_frames.append(sub)

wide_df = pivot_frames[0]
for frame in pivot_frames[1:]:
    wide_df = wide_df.merge(frame.drop(columns=["true_label"]), on="filepath", how="inner")

print(f"Total gambar dgn OOF prediction dari SEMUA model: {len(wide_df)} (dari total {len(full_df)} gambar)")
if len(wide_df) < len(full_df):
    print(f"  [INFO] {len(full_df) - len(wide_df)} gambar tidak dipakai -- cek konsistensi antar model jika ini besar.")

wide_df.to_csv(OUTPUT_DIR / "oof_predictions_wide.csv", index=False)
print(f"[SAVED] oof_predictions_wide.csv")
wide_df.head()

## Langkah Selanjutnya (Tahap 2-5 workflow stacking)
1. **Tahap 2 — Latih meta-learner**: pakai `oof_predictions_wide.csv`, latih Logistic Regression dari 9 kolom probabilitas sebagai fitur, evaluasi dgn CV kecil.
2. **Tahap 3 — Retrain final models**: latih ulang 3 model ini pakai 100% data train (bukan per-fold).
3. **Tahap 4 — Prediksi ke Data Uji**: 1.458 gambar test → probabilitas dari 3 model final → masuk ke meta-learner.
4. **Tahap 5 — Submission**: format sesuai aturan kompetisi, siap upload.